# Cahoy 2010 vs Aurora PICASO comparison

Run after the two-stage Cahoy replication completes (`304` NetCDF spectra).

```bash
source env/activate_aurora_picaso4_job.sh
bash roadrunner_egp/aurora_subneptune_grid/scripts/install_cahoy2010_reference.sh
```

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "roadrunner_egp").exists():
    REPO_ROOT = Path.cwd().parents[1]

GRID_ROOT = REPO_ROOT / "roadrunner_egp" / "aurora_subneptune_grid"
MODEL = "aurora_cahoy2010_replication_v0"
MANIFEST = GRID_ROOT / "manifests" / f"{MODEL}_manifest.csv"
NC_ROOT = GRID_ROOT / "outputs" / MODEL / "nc"
COMPARE_ROOT = GRID_ROOT / "outputs" / MODEL / "cahoy_compare"
PLOT_ROOT = COMPARE_ROOT / "plots"

for extra in (GRID_ROOT / "src", GRID_ROOT.parent):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from aurora_grid.cahoy_compare import (
    compare_manifest_outputs,
    compare_nc_to_cahoy,
    ensure_reference_installed,
    metrics_to_records,
)
from aurora_grid.cahoy_reference import resolve_reference_root

REFERENCE_ROOT = ensure_reference_installed()
COMPARE_ROOT.mkdir(parents=True, exist_ok=True)
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repo: {REPO_ROOT}")
print(f"Reference: {REFERENCE_ROOT}")
print(f"NC dir: {NC_ROOT}")
print(f"NC count: {len(list(NC_ROOT.glob('run_*.nc')))}")

In [ ]:
results = compare_manifest_outputs(MANIFEST, NC_ROOT, reference_root=REFERENCE_ROOT)
records = metrics_to_records(results)
df = pd.DataFrame(records)
df_ok = df[df["status"] == "ok"].copy()
print(f"compared: {len(df_ok)} / {len(df)} rows")
if not df_ok.empty:
    display(df_ok.sort_values("rmse", ascending=False).head(10))
else:
    display(df.head())

In [ ]:
metrics_path = COMPARE_ROOT / "cahoy_compare_metrics.csv"
df.to_csv(metrics_path, index=False)
summary = {
    "n_compared": int((df["status"] == "ok").sum()),
    "n_missing_nc": int((df["status"] == "missing_nc").sum()),
    "median_rmse": float(df_ok["rmse"].median()) if not df_ok.empty else None,
}
(COMPARE_ROOT / "cahoy_compare_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(f"wrote {metrics_path}")

In [ ]:
def plot_pair(metrics, arrays, out_path=None):
    wave = arrays["wavelength_um"]
    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    axes[0].plot(wave, arrays["cahoy_albedo"], label="Cahoy 2010", color="0.25")
    axes[0].plot(wave, arrays["aurora_albedo"], label="Aurora PICASO", color="C0")
    axes[0].set_ylabel("Albedo")
    axes[0].set_title(metrics.cahoy_reference_name)
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].plot(wave, arrays["residual"], color="C3")
    axes[1].axhline(0, color="0.4", lw=0.8)
    axes[1].set_xlabel("Wavelength (um)")
    axes[1].set_ylabel("Aurora - Cahoy")
    axes[1].grid(alpha=0.25)
    fig.suptitle(f"RMSE={metrics.rmse:.4f}  phase={metrics.phase_deg:.0f} deg", fontsize=10)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
    return fig

ok = [item for item in results if item[2] is None]
for metrics, arrays, _ in sorted(ok, key=lambda item: item[0].rmse, reverse=True)[:6]:
    stem = metrics.cahoy_reference_name.replace(".dat", "")
    plot_pair(metrics, arrays, PLOT_ROOT / f"{stem}.png")
    plt.show()

In [ ]:
# Pick one case to inspect interactively
CASE_NAME = "Jupiter_1x_0.8AU_0deg.dat"
nc_files = sorted(NC_ROOT.glob("run_*.nc"))
if nc_files:
    for nc in nc_files[:5]:
        metrics, arrays = compare_nc_to_cahoy(nc, reference_root=REFERENCE_ROOT)
        if metrics.cahoy_reference_name == CASE_NAME:
            plot_pair(metrics, arrays)
            plt.show()
            break
else:
    print("No NetCDF files yet — rerun this notebook tomorrow after stage 2 completes.")